# 01 — Data Ingestion Pipeline

This notebook pulls NFL player data from the **Sleeper API**, caches it locally,
applies position filtering, and merges it with a local projections CSV.

**Workflow**
1. Environment & path setup
2. Sleeper API extraction & caching
3. Parse & filter the Sleeper catalog
4. Normalization & projections ingestion
5. Merging & audit verification

## Cell 1 — Environment & Setup

In [8]:
import os
import re
import json
import requests
import pandas as pd

# -- Project paths --------------------------------------------------
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR     = os.path.join(PROJECT_ROOT, "data")
NOTEBOOK_DIR = os.path.join(PROJECT_ROOT, "notebooks")

SLEEPER_CACHE   = os.path.join(DATA_DIR, "sleeper_players_raw.json")
PROJECTIONS_CSV = os.path.join(DATA_DIR, "projections_template.csv")

SKILL_POSITIONS = ["QB", "RB", "WR", "TE"]

os.makedirs(DATA_DIR, exist_ok=True)
print(f"DATA_DIR      -> {DATA_DIR}")
print(f"SLEEPER_CACHE -> {SLEEPER_CACHE}")
print(f"PROJECTIONS   -> {PROJECTIONS_CSV}")

DATA_DIR      -> /home/hadev/Projects/Lab/fantasy-footbal-analytics/data
SLEEPER_CACHE -> /home/hadev/Projects/Lab/fantasy-footbal-analytics/data/sleeper_players_raw.json
PROJECTIONS   -> /home/hadev/Projects/Lab/fantasy-footbal-analytics/data/projections_template.csv


## Cell 2 — Sleeper API Extraction & Caching

Pull the full NFL player catalog from the Sleeper public API and persist it as
raw JSON. If the cache file already exists the download is skipped unless
`force_refresh=True` is set.

In [9]:
SLEEPER_API_URL = "https://api.sleeper.app/v1/players/nfl"

def fetch_sleeper_players(force_refresh: bool = False) -> dict:
    """Download (or load from cache) the full Sleeper NFL player catalog."""
    if not force_refresh and os.path.exists(SLEEPER_CACHE):
        print(f"Loading cached players from {SLEEPER_CACHE}")
        with open(SLEEPER_CACHE, "r") as f:
            return json.load(f)

    print("Fetching players from Sleeper API ...")
    resp = requests.get(SLEEPER_API_URL, timeout=60)
    resp.raise_for_status()
    players = resp.json()

    os.makedirs(os.path.dirname(SLEEPER_CACHE), exist_ok=True)
    with open(SLEEPER_CACHE, "w") as f:
        json.dump(players, f)
    print(f"Cached {len(players):,} players -> {SLEEPER_CACHE}")
    return players

sleeper_raw = fetch_sleeper_players(force_refresh=False)
print(f"Total player records: {len(sleeper_raw):,}")

Loading cached players from /home/hadev/Projects/Lab/fantasy-footbal-analytics/data/sleeper_players_raw.json
Total player records: 12,223


## Cell 3 — Parse & Filter Sleeper Catalog

From the raw catalog we keep only **active** players at skill positions and
extract the columns relevant to downstream merging and analytics.

In [10]:
def parse_sleeper_catalog(raw_data: dict, positions: list) -> pd.DataFrame:
    """
    Parse raw Sleeper JSON into a clean DataFrame, filtering for active
    players at the specified skill positions.
    """
    records = []
    for pid, p in raw_data.items():
        if (
            p.get("position") in positions
            and p.get("status", "").lower() == "active"
        ):
            records.append({
                "player_id":      p.get("player_id", pid),
                "full_name":      p.get("full_name"),
                "position":       p.get("position"),
                "team":           p.get("team"),
                "search_rank":    p.get("search_rank"),
                "years_exp":      p.get("years_exp"),
                "status":         p.get("status"),
                "injury_status":  p.get("injury_status"),
            })

    df = pd.DataFrame(records)
    df.drop_duplicates(subset=["player_id"], inplace=True)
    return df

sleeper_df = parse_sleeper_catalog(sleeper_raw, SKILL_POSITIONS)

print(f"Active skill-position players: {len(sleeper_df):,}")
print(f"\nPosition breakdown:")
print(sleeper_df["position"].value_counts().to_string())
print(f"\nSample rows (top 10 by search_rank):")
sleeper_df.sort_values("search_rank").head(10)

Active skill-position players: 2,771

Position breakdown:
position
WR    1245
RB     620
TE     569
QB     337

Sample rows (top 10 by search_rank):


,player_id,full_name,position,team,search_rank,years_exp,status,injury_status
1290,9509,Bijan Robinson,RB,ATL,1.0,3,Active,NaN
1387,9221,Jahmyr Gibbs,RB,DET,1.0,3,Active,NaN
519,7564,Ja'Marr Chase,WR,CIN,4.0,5,Active,NaN
2124,4984,Josh Allen,QB,BUF,4.0,8,Active,NaN
2697,6813,Jonathan Taylor,RB,IND,4.0,6,Active,NaN
154,8138,James Cook,RB,BUF,5.0,4,Active,NaN
948,9493,Puka Nacua,WR,LAR,5.0,3,Active,Questionable
988,4034,Christian McCaffrey,RB,SF,5.0,9,Active,Questionable
252,9488,Jaxon Smith-Njigba,WR,SEA,6.0,3,Active,NaN
931,3198,Derrick Henry,RB,BAL,7.0,10,Active,NaN


## Cell 4 — Normalization & Projections Ingestion

A robust string-normalization helper strips punctuation, lowercases, and removes
common name suffixes (Jr., Sr., III, etc.) so that player names from different
sources can be reliably joined.

In [11]:
# Suffixes to strip (matched as whole trailing tokens)
SUFFIX_PATTERN = re.compile(
    r'\b(jr|sr|ii|iii|iv|v|esq|phd)\b\s*$',
    re.IGNORECASE,
)

def clean_player_name(name: str) -> str:
    """
    Normalize a player name for fuzzy matching:
      - lowercase
      - remove apostrophes, hyphens, periods
      - strip trailing suffixes (Jr., Sr., III, IV, etc.)
      - collapse whitespace
    """
    if not isinstance(name, str):
        return ""
    text = name.lower()
    text = text.replace("'", "").replace("-", "").replace(".", "")
    text = SUFFIX_PATTERN.sub("", text)
    text = " ".join(text.split())  # collapse whitespace
    return text.strip()

# Quick sanity check
test_names = [
    "Ja'Marr Chase",
    "Kenneth Walker III",
    "Marvin Harrison Jr.",
    "Travis Kelce",
]
for n in test_names:
    print(f"  {n!r:30s} -> {clean_player_name(n)!r}")

  "Ja'Marr Chase"                -> 'jamarr chase'
  'Kenneth Walker III'           -> 'kenneth walker'
  'Marvin Harrison Jr.'          -> 'marvin harrison'
  'Travis Kelce'                 -> 'travis kelce'


In [12]:
projections_df = pd.read_csv(PROJECTIONS_CSV)

# Apply normalization columns
projections_df["norm_name"]    = projections_df["player_name"].apply(clean_player_name)
projections_df["norm_position"] = projections_df["position"].str.strip().str.upper()

# Do the same for the Sleeper side
sleeper_df["norm_name"]    = sleeper_df["full_name"].apply(clean_player_name)
sleeper_df["norm_position"] = sleeper_df["position"].str.strip().str.upper()

print(f"Projections loaded: {len(projections_df)} rows")
print(f"Columns: {list(projections_df.columns)}")
projections_df

Projections loaded: 1032 rows
Columns: ['player_name', 'position', 'team', 'proj_points', 'norm_name', 'norm_position']


,player_name,position,team,proj_points,norm_name,norm_position
0,L.Jackson,QB,BAL,428.4,ljackson,QB
1,J.Chase,WR,CIN,403.0,jchase,WR
2,J.Allen,QB,BUF,400.6,jallen,QB
3,J.Burrow,QB,CIN,372.8,jburrow,QB
4,S.Barkley,RB,PHI,371.1,sbarkley,RB
...,...,...,...,...,...,...
1027,K.Trask,QB,TB,-0.6,ktrask,QB
1028,T.Bagent,QB,CHI,-0.7,tbagent,QB
1029,K.Toney,WR,CLE,-1.1,ktoney,WR
1030,D.Laube,RB,LV,-5.7,dlaube,RB


## Cell 5 — Merging & Audit Verification

We perform both an **inner** join (to see confirmed matches) and a **left** join
(to identify projection players that could not be matched to a Sleeper record).

In [13]:
MERGE_KEYS = ["norm_name", "norm_position"]

# -- Inner join: confirmed matches -----------------------------------
matched_df = pd.merge(
    projections_df,
    sleeper_df,
    on=MERGE_KEYS,
    how="inner",
    suffixes=("_proj", "_sleeper"),
)

print(f"=== Merge Summary ===")
print(f"Projections rows:  {len(projections_df)}")
print(f"Sleeper rows:      {len(sleeper_df):,}")
print(f"Inner-join matches: {len(matched_df)}")
print(f"Match rate: {len(matched_df)/len(projections_df)*100:.1f}%")

print(f"\n=== Matched Players ===")
result = matched_df[["player_name", "position_proj", "team_proj", "proj_points",
           "player_id", "full_name", "team_sleeper", "search_rank"]]
result = result.sort_values("proj_points", ascending=False)
result

=== Merge Summary ===
Projections rows:  1032
Sleeper rows:      2,771
Inner-join matches: 531
Match rate: 51.5%

=== Matched Players ===


,player_name,position_proj,team_proj,proj_points,player_id,full_name,team_sleeper,search_rank
0,Josh Allen,QB,BUF,360.0,4984,Josh Allen,BUF,4.0
1,Drake Maye,QB,NE,320.0,11564,Drake Maye,NE,8.0
2,Puka Nacua,WR,LAR,310.0,9493,Puka Nacua,LAR,5.0
3,Ja'Marr Chase,WR,CIN,310.0,7564,Ja'Marr Chase,CIN,4.0
4,Bijan Robinson,RB,ATL,290.0,9509,Bijan Robinson,ATL,1.0
...,...,...,...,...,...,...,...,...
526,Hunter Henry,TE,NE,110.0,3214,Hunter Henry,NE,92.0
527,Teagan Quitoriano,TE,ARI,110.0,8227,Teagan Quitoriano,ARI,264.0
528,Oronde Gadsden,TE,LAC,110.0,12493,Oronde Gadsden,LAC,83.0
529,Kyle Pitts,TE,ATL,110.0,7553,Kyle Pitts,ATL,70.0


In [14]:
# -- Audit: unmatched projection players -------------------------------
left_join_df = pd.merge(
    projections_df,
    sleeper_df,
    on=MERGE_KEYS,
    how="left",
    indicator=True,
    suffixes=("_proj", "_sleeper"),
)

unmatched = left_join_df[left_join_df["_merge"] == "left_only"].copy()

print(f"\n=== Audit Report ===")
print(f"Matched:    {len(left_join_df[left_join_df['_merge'] == 'both']):>4d}")
print(f"Unmatched:  {len(unmatched):>4d}")
print(f"Total:      {len(projections_df):>4d}")

if len(unmatched) > 0:
    print(f"\nUnmatched projection players (need manual review):")
    display(unmatched[["player_name", "position_proj", "team_proj", "proj_points", "norm_name"]])
else:
    print(f"\nAll projection players matched successfully!")


=== Audit Report ===
Matched:     531
Unmatched:   502
Total:      1032

Unmatched projection players (need manual review):


,player_name,position_proj,team_proj,proj_points,norm_name
0,L.Jackson,QB,BAL,428.4,ljackson
1,J.Chase,WR,CIN,403.0,jchase
2,J.Allen,QB,BUF,400.6,jallen
3,J.Burrow,QB,CIN,372.8,jburrow
4,S.Barkley,RB,PHI,371.1,sbarkley
...,...,...,...,...,...
1028,K.Trask,QB,TB,-0.6,ktrask
1029,T.Bagent,QB,CHI,-0.7,tbagent
1030,K.Toney,WR,CLE,-1.1,ktoney
1031,D.Laube,RB,LV,-5.7,dlaube


---

**Next steps:**
- Replace `projections_template.csv` with real projection data.
- Add ADP, injury, or bye-week data sources in subsequent notebooks.
- Use the merged dataset for draft value calculations in `02_draft_analysis.ipynb`.